# Inference Notebook

The below cell can be edited to provide an input sentence to the models we have trained. A few possible strings are provided that have been taken from the dataset, but any sentence in the correct format can be used.

The sentence should have two 'entities', each enclosed with <e1></e1> and <e2></e2> tags respectively. 

In [15]:
InputSentence = 'The <e1>apples</e1> are in the <e2>basket</e2>.'
InputSentence = 'The House has postponed the <e1>vote</e1> to <e2>tomorrow</e2>.'
InputSentence = 'The animal control released a wild <e1>leopard</e1> into the <e2>wild</e2>.'
InputSentence = 'Approximately $2 <e1>trillions</e1> are invested into SRI Socially responsible investing <e2>ventures</e2> in the present day.'

#### WARNING - NO CODE BELOW THIS POINT SHOULD BE EDITED

The cell below simply tokenises the InputSentence to a list of tokens, ensuring the tags are kept together. This is necessary as input for the final two models.

In [16]:
from nltk.tokenize import word_tokenize

sentence = InputSentence.replace("<e1>", " e1start ").replace("</e1>", " e1end ")
sentence = sentence.replace("<e2>", " e2start ").replace("</e2>", " e2end ")
InputSentenceSeparated = word_tokenize(sentence)
InputSentenceSeparated = ["<e1>" if item == "e1start" else item for item in InputSentenceSeparated]
InputSentenceSeparated = ["<e2>" if item == "e2start" else item for item in InputSentenceSeparated]
InputSentenceSeparated = ["</e1>" if item == "e1end" else item for item in InputSentenceSeparated]
InputSentenceSeparated = ["</e1>" if item == "e2end" else item for item in InputSentenceSeparated]
print(InputSentenceSeparated)

['The', 'House', 'has', 'postponed', 'the', '<e1>', 'vote', '</e1>', 'to', '<e2>', 'tomorrow', '</e1>', '.']


The cell below imports the necessary packages for the notebook to run. Please ensure the packages are installed and are imported correctly.

In [17]:
#Importing required libraries
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torch.nn import init
import torch.nn.functional as F
import pandas as pd
import random
import warnings
warnings.filterwarnings('ignore')

The cell below enables GPU processing if possible. It also defines a function used to extract the textual form of a predicted label from the models. For example, the models will take a sentence and return an integer, this function takes that integer and gives the original relation in english text.

In [18]:
use_cuda = torch.cuda.is_available()
device = torch.device("cuda:0" if use_cuda else "cpu")
#device = torch.device("cpu")

def get_relation_from_id(relation_id):
    match relation_id:
        case 0: return "Cause-Effect(e1,e2)"
        case 1: return "Cause-Effect(e2,e1)"
        case 2: return "Component-Whole(e1,e2)"
        case 3: return "Component-Whole(e2,e1)"
        case 4: return "Content-Container(e1,e2)"
        case 5: return "Content-Container(e2,e1)"
        case 6: return "Entity-Destination(e1,e2)"
        case 7: return "Entity-Destination(e2,e1)"
        case 8: return "Entity-Origin(e1,e2)"
        case 9: return "Entity-Origin(e2,e1)"
        case 10: return "Instrument-Agency(e1,e2)"
        case 11: return "Instrument-Agency(e2,e1)"
        case 12: return "Member-Collection(e1,e2)"
        case 13: return "Member-Collection(e2,e1)"
        case 14: return "Message-Topic(e1,e2)"
        case 15: return "Message-Topic(e2,e1)"
        case 16: return "Product-Producer(e1,e2)"
        case 17: return "Product-Producer(e2,e1)"
        case 18: return "Other"
    return "ERROR - relation_id NOT VALID!!"

Below are our 4 models. Each model has a heading, and a single cell with the code required to run the model on the supplied InputSentence defined above.

They each run and finally print the output of the model as text. If there are any errors, please ensure the required embedding, model, and code files are accessible to this notebook, as described in the README.md file.

## Base Bi-LSTM Model

In [19]:
from Base_BLSTM import RelationClassification_Model, Vocabulary_and_Dataframe, test_model

train_df = pd.read_parquet('train-00000-of-00001.parquet', engine='pyarrow')
test_df = pd.read_parquet('test-00000-of-00001.parquet', engine='pyarrow')

d = {'sentence': [InputSentence], 'relation': 0}
sentence_df = pd.DataFrame(data=d)

vocab_df = Vocabulary_and_Dataframe(freq_threshold=5)

vocab_df.build_vocabulary_and_dataframe(train_df)
vocab_df.build_vocabulary_and_dataframe(test_df)
vocab_df.build_vocabulary_and_dataframe(sentence_df)
vocab_size = len(vocab_df)
pad_idx = vocab_df.str_2_idx[""]

emb_size = 100
hidden = 50
layers = 1
class_num = 19
lr = 0.005
wd = 0.0001

test_indices = sentence_df.index.values
partition = {
    'test': test_indices
}

model = RelationClassification_Model(emb_size, hidden, vocab_size, layers, class_num, pad_idx)
model.load_state_dict(torch.load('model_base.pkl', weights_only=True))
model.to(device)
model.eval()

batch_size = 1
testing_stats = test_model(sentence_df, partition, vocab_df, device, model, batch_size)
print("Relation: ", get_relation_from_id(testing_stats[-1]))

FileNotFoundError: [Errno 2] No such file or directory: 'model_base.pkl'

## Glove Enhanced Bi-LSTM Model

In [5]:
from GloveEmbeddings_BLSTM import load_glove_embeddings

embedding_size = 300
hidden_size = 100
num_layers = 2
learning_rate = 0.005
weight_decay= 0.0001
num_epochs = 100

print("Loading Glove embeddings...")
glove_embeddings = load_glove_embeddings(embedding_size, vocab_df)

model = RelationClassification_Model(embedding_size, hidden_size, vocab_size, num_layers, class_num, pad_idx)
model.embedding = nn.Embedding.from_pretrained(torch.tensor(glove_embeddings, dtype=torch.float), freeze=False)
model.load_state_dict(torch.load('model_glove.pkl', weights_only=True))
model.to(device)
model.eval()

batch_size = 1
testing_stats = test_model(sentence_df, partition, vocab_df, device, model, batch_size)
print("Relation: ", get_relation_from_id(testing_stats[-1]))


Loading Glove embeddings...
Relation:  Other


## Attention Enhanced Bi-LSTM Model

In [30]:
from Att_BLSTM import WordEmbeddingLoader, RelationLoader, Att_BLSTM, preprocess_sentence, predict_label

word2id, word_vec = WordEmbeddingLoader().load_embedding()
rel2id, id2rel, class_num = RelationLoader().get_relation()

model = Att_BLSTM(word_vec=word_vec, class_num=class_num)
model.load_state_dict(torch.load('model_att.pkl', weights_only=True))
model = model.to(device)
model.eval()

input = preprocess_sentence(InputSentenceSeparated, word2id).to(device)

print("Relation: ", get_relation_from_id(predict_label(model, input)))

Content-Container(e2,e1)


## Multi-Head Attention Enhanced Bi-LSTM Model

In [31]:
from MultiHeadAtt_BLSTM import WordEmbeddingLoader, RelationLoader, MultiHeadAtt_BLSTM, preprocess_sentence, predict_label

word2id, word_vec = WordEmbeddingLoader().load_embedding()
rel2id, id2rel, class_num = RelationLoader().get_relation()

model = MultiHeadAtt_BLSTM(word_vec=word_vec, class_num=class_num)
model.load_state_dict(torch.load('model_multiheadatt.pkl', weights_only=True))
model = model.to(device)
model.eval()

input = preprocess_sentence(InputSentenceSeparated, word2id).to(device)

print("Relation: ", get_relation_from_id(predict_label(model, input)))

Relation:  Content-Container(e2,e1)
